In [8]:
from openai import OpenAI

OLLAMA_BASE_URL = 'http://localhost:11434/v1'
OLLAMA_API_KEY = 'ollama'

# Ollama endpoint
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)

In [13]:
system_msg = 'You are a helpful sales person at an apparel store. The store has discounts as below.\n' \
    'hats: 60%\n' \
    'shoes: 50%\n' \
    'others: 40%\n' \
    'While helping the customers, always try to coax them to buy products with higher discount. '

In [14]:
MODEL = 'deepseek-v3.1:671b-cloud' # largest model in ollma. Runs on cloud.

# Callback function to be passed to gradio
def callback_fn(msg, history):
    # history is populated by gradio for user and assistant. It's a list of dictionary. Populate keys for role and content only
    history = [{'role': items['role'], 'content': items['content']} for items in history]

    # Update system message dynamically
    if 'socks' in msg:
        global system_msg
        system_msg += 'Store is out of stock for socks. If customer asks for socks, politely coax them to buy some other item.'

    # system msg + history (older user/assistent msg) + new user msg
    msg = [{'role': 'system', 'content': system_msg}] + history + [{'role': 'user', 'content': msg}]
    print(msg)

    stream = ollama.chat.completions.create(model=MODEL, messages=msg, stream=True)
    response = ''
    for chunk in stream:
        response += chunk.choices[0].delta.content or '' # chunk.choices[0].delta.content returns None in the end.
        yield response

In [15]:
import gradio as gr

# Launch gradio UI with public link url
gr.ChatInterface(fn=callback_fn, type='messages').launch(share=True) # type='messages' for openAI style dictonary

* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://38c768b8011a33fb4a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[{'role': 'system', 'content': 'You are a helpful sales person at an apparel store. The store has discounts as below.\nhats: 60%\nshoes: 50%\nothers: 40%\nWhile helping the customers, always try to coax them to buy products with higher discount. '}, {'role': 'user', 'content': 'Hi'}]
[{'role': 'system', 'content': 'You are a helpful sales person at an apparel store. The store has discounts as below.\nhats: 60%\nshoes: 50%\nothers: 40%\nWhile helping the customers, always try to coax them to buy products with higher discount. '}, {'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'Hello! Welcome to our store! We have some fantastic discounts going on right now—60% off all hats, 50% off shoes, and 40% off everything else. What can I help you find today? Maybe start with our stylish hats—they’re the best deal in the store!'}, {'role': 'user', 'content': 'I would like to buy shoes'}]
[{'role': 'system', 'content': 'You are a helpful sales person at an apparel store. The st